# Dataset Preparation

Prep based on: https://github.com/kerrgarr/SemanticSegmentationCityscapes

Download Cityscapes datasets: https://www.cityscapes-dataset.com/downloads

Datasets used: gtFine_trainvaltest.zip (241MB), leftmg8bit_trainvaltest.zip (11GB)

Unzip both files in the data/ directory

Your data/ folder structure should look like:

data/   
├──gtFine_trainvaltest/   
└──leftImg8bit_trainvaltest/    

In [ ]:
import os
import shutil
from tqdm import tqdm

In [ ]:
# new dataset
DATASET_DIR = '../data/dataset/'
# original datasets
LABELS_DIR = '../data/gtFine_trainvaltest/gtFine/'
IMAGES_DIR = '../data/leftImg8bit_trainvaltest/leftImg8bit/'

In [ ]:
# See available cities in the dataset
for folder in os.listdir(IMAGES_DIR):
    cities = []
    for city in os.listdir(os.path.join(IMAGES_DIR, folder)):
        cities.append(city)
    print(f"{folder}: {cities}")

In [ ]:
# See how many images are in each city folder
for folder in os.listdir(IMAGES_DIR):
    print(folder)
    for city in os.listdir(os.path.join(IMAGES_DIR, folder)):
        path = os.path.join(IMAGES_DIR, folder, city)
        print(f"  {city}: {len(os.listdir(path))}")

**NOTE** - The official Cityscapes test split does not contian image masks, only blanks. The test split is used by the official Cityscapes benchmark to evaluate performance. The original test cannot be used to evaluate model accuracy locally. 

Use some data from the original train split to create test split, but exclude it.

In [ ]:
# Train directory
gtFineDIR_train = os.path.join(LABELS_DIR, 'train')
ImgDIR_train = os.path.join(IMAGES_DIR, 'train')

dataset_train = os.path.join(DATASET_DIR,'train')

# Validation directory
gtFineDIR_val = os.path.join(LABELS_DIR, 'val')
ImgDIR_val = os.path.join(IMAGES_DIR, 'val')

dataset_valid = os.path.join(DATASET_DIR,'val')

# Test directory
dataset_test = os.path.join(DATASET_DIR, 'test')

In [ ]:
# copy images and labels per cities into one target directory
def create_splits(sourcedir, targetdir, cities_list):
    extensions = ['labelIds', 'leftImg8bit'] # only labels and images needed
    os.makedirs(targetdir, exist_ok=True)
    
    for city in tqdm(cities_list, desc="Cities"):
        directory_city = os.path.join(sourcedir, city)
        images = os.listdir(directory_city)
        for image in images:
            if any(word in image for word in extensions):
                image_path = os.path.join(directory_city, image)
                try:
                    shutil.copy(image_path, targetdir)
                except OSError as e:
                    tqdm.write(str(e))

Select cities to use in each split.

Keep entire cities per split instead of randomly selecting images from all cities. This is to prevent data leakage from similar looking frames and to improve generalization.

Copy images and matching label masks into splits.

In [ ]:
train_cities = ['aachen', 'bochum', 'bremen', 'cologne', 'darmstadt', 'erfurt', 'hamburg', 'hanover', 'krefeld', 'monchengladbach', 'strasbourg', 'tubingen', 'ulm', 'weimar', 'zurich']

create_splits(gtFineDIR_train, dataset_train, train_cities)
create_splits(ImgDIR_train, dataset_train, train_cities)

In [ ]:
valid_cities = ['frankfurt', 'lindau', 'munster']

create_splits(gtFineDIR_val, dataset_valid, valid_cities)
create_splits(ImgDIR_val, dataset_valid, valid_cities)

In [ ]:
# NOTE - These cities are from the original train split. We do not include them in our train split!
test_cities = ['dusseldorf', 'jena', 'stuttgart']

create_splits(gtFineDIR_train, dataset_test, test_cities)
create_splits(ImgDIR_train, dataset_test, test_cities)

In [ ]:
# Roughly 70 / 15 / 15 split
for folder in os.listdir(DATASET_DIR):
    path = os.path.join(DATASET_DIR, folder)
    print(folder, len(os.listdir(path))//2) # divide by 2 because each image has a matching mask file